# MODELO ESTIMACIÓN PROBABILIDAD DE REVERSIÓN GRANDES FUTUROS CRIPTOMONEDAS - BINANCE

In [1]:
# =========================================================
# LIBRERÍAS
# =========================================================
import requests
import os
import pandas as pd
import numpy as np
import ta
import joblib

import time
from datetime import datetime, timezone, timedelta

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore")

import winsound
import time

# 0. ENTRENAMIENTO DEL MODELO

## 🎯 Objetivo del modelo

Detectar reversiones de sobrecompra / sobreventa en 1H, con resultado en en al menos MIN_HOLD_BARS velas de 15 minutos.

Reversiones fuertes en al menos MIN_HOLD_BARS velas de 15 minutos, con confirmación posterior en 15m.

👉 Esto implica que NO necesitas años de datos, necesitas suficientes ciclos completos de mercado.

Función y Propósito

get_all_usdt_futures_symnbols: Obtiene el universo de símbolos sobre los que el modelo puede operar.

filter_symbols_pre_ml: Filtra símbolos antes del scanner ML. Selecciona criptos líquidas, volátiles y operables para reversiones duraderas.

get_ohlc: Descargar velas OHLCV limpias y confiables desde Binance.

add_features: Calcular features de ML coherentes con el entrenamiento.

build_dataset: Construir el dataset de entrenamiento del modelo ML.

train_model: Entrenar el modelo ML de probabilidad de reversión.

scan_market_ml: Detectar símbolos candidatos a reversión (1H).

df15_provider: Proveer velas 15m con suficiente contexto para análisis secuencial.

atr_15m_provider: Calcular ATR realista y estable para gestión de riesgo.

confirm_entry_15m_realtime: Esperar y confirmar el momento exacto de entrada en velas de 15 minutos.

structural_exit_conditions: Detectar cuando el contexto de reversión deja de ser válido.

monitor_trade: Gestionar TODO el ciclo de vida del trade con alertas.


Diseño final del pipeline (mental model)

get_ohlc

Descarga OHLC crudo (endpoint REST, sin cliente Binance)

add_features

ATR, retornos, volatilidad, etc.

filter_symbols_pre_ml

Selección previa de símbolos “entrenables”

build_dataset

Itera velas

Calcula movimientos futuros

Aplica reglas de reversión

Construye labels y dataset final

ML training / validation

In [2]:
# =========================================================
# CONFIGURACIÓN GLOBAL
# =========================================================
BINANCE_FUTURES_BASE = "https://fapi.binance.com"


HORIZON_BARS = 4          # 1h = 4 velas de 15m

MIN_MOVE_ATR = 1.0
MIN_HOLD_BARS = 1
MAX_ADVERSE_ATR = 0.8

INTERVAL = "1h"
LOOKBACK = 120    # Longitud de Periodo a evaluar en timeframe de variable INTERVAL para seleccionar las cripto más representativas para el entrenamiento del modelo
MAX_SYMBOLS_TRAIN = 70       # limitar universo de entrenamiento





#
#TRAIN_SIZE = 2000                  # Número de velas a descargar para entrenar el modelo (cantidad de datapoints)
#SCAN_SIZE = 120
#horizon = 4                  # horas futuras
#REV_PCT = 0.04               # 4% reversión


GMT5 = timezone(timedelta(hours=-5))
GMT = timezone(timedelta(hours=0))






# =========================================================
# 1️⃣ UNIVERSO BINANCE FUTURES
# =========================================================
def get_all_usdt_futures_symbols():
    """
    Retorna todos los símbolos de futuros perpetuos USDT en Binance.
    """
    response = requests.get(
        f"{BINANCE_FUTURES_BASE}/fapi/v1/exchangeInfo",
        timeout=10
    )
    response.raise_for_status()

    data = response.json()

    return [
        s["symbol"] for s in data["symbols"]
        if s["contractType"] == "PERPETUAL"
        and s["quoteAsset"] == "USDT"
        and s["status"] == "TRADING"
    ]


# =========================================================
# 1️⃣.1 FILTRANDO UNIVERSO BINANCE FUTURES PARA OBTENER LOS SÍMBOLOS ESTABLES, LÍQUIDOS Y REPRESENTATIVOS PARA ENTRENAMIENTO DEL MODELO
# =========================================================
def filter_symbols_pre_ml(
    symbols,
    interval=INTERVAL,
    lookback=LOOKBACK,
    min_notional_volume=5_000_000,
    min_atr_rel=0.0015,
    max_atr_rel=0.02,
    max_symbols=MAX_SYMBOLS_TRAIN
):
    """
    Filtra símbolos antes del scanner ML.
    Selecciona criptos líquidas, volátiles y operables
    para reversiones duraderas.
    """

    qualified = []

    for symbol in symbols:
        try:
            df = get_ohlc(symbol, interval=interval, limit=lookback)
            if df is None or len(df) < lookback:
                continue

            # --- Volumen nocional ---
            df["notional"] = df["close"] * df["volume"]
            avg_notional = df["notional"].mean()

            if avg_notional < min_notional_volume:
                continue

            # --- ATR relativo ---
            df["atr"] = ta.volatility.average_true_range(
                df["high"], df["low"], df["close"], window=14
            )
            atr_rel = (df["atr"] / df["close"]).mean()

            if not (min_atr_rel <= atr_rel <= max_atr_rel):
                continue

            # --- Estabilidad (evita pumps aislados) ---
            avg_range = (df["high"] - df["low"]).mean()
            range_std = (df["high"] - df["low"]).std()

            if range_std / avg_range > 1.8:
                continue

            qualified.append({
                "Symbol": symbol,
                "ATR_rel": round(atr_rel, 5),
                "Avg_Notional": int(avg_notional)
            })

        except Exception:
            continue

    if not qualified:
        return []

    dfq = pd.DataFrame(qualified)

    # Ranking por calidad operativa
    dfq = dfq.sort_values(
        ["Avg_Notional", "ATR_rel"],
        ascending=[False, False]
    )

    return dfq.head(max_symbols)["Symbol"].tolist()




# =========================================================
# 2️⃣ OHLC
# =========================================================
def get_ohlc(
    symbol: str,
    interval: str = "1h",
    limit: int = 500,
    max_total: int | None = None
):
    """
    Descarga OHLCV desde Binance Futures (USDT-M).
    
    - Usa requests (sin API key)
    - Soporta paginación si max_total > limit
    - Devuelve DataFrame limpio y ordenado
    """

    url = f"{BINANCE_FUTURES_BASE}/fapi/v1/klines"
    all_data = []
    end_time = None

    max_total = max_total or limit

    while len(all_data) < max_total:
        fetch = min(1500, max_total - len(all_data))

        params = {
            "symbol": symbol,
            "interval": interval,
            "limit": fetch
        }

        if end_time is not None:
            params["endTime"] = end_time

        try:
            r = requests.get(url, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            
            if not data:
                break

            all_data = data + all_data
            end_time = data[0][0] - 1  # ms, vela anterior

        except Exception:
            break

    if len(all_data) < 50:
        return None

    df = pd.DataFrame(
        all_data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "num_trades",
            "taker_buy_base_volume",
            "taker_buy_quote_volume",
            "ignore"
        ]
    )

    # ---------- Tipos ----------
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # ---------- Timestamps ----------
    df["time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)

    df = df[["time", "open", "high", "low", "close", "volume"]]
    df.sort_values("time", inplace=True)
    df.reset_index(drop=True, inplace=True)

    df.dropna(inplace=True)

    if len(df) < 50:
        return None

    return df






# =========================================================
# 3️⃣ INDICADORES TÉCNICOS (SIMÉTRICOS LONG/SHORT)
# =========================================================
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula FEATURES ESTRUCTURALES para el modelo ML.

    Diseñada exclusivamente para:
    - Detectar REVERSIÓN GRANDE y DURADERA
    - Escenarios de 1H (scanner / entrenamiento)

    NO se usa para:
    - Confirmación de entrada
    - Timing
    - Gestión de la orden
    """

    df = df.copy()

    # =============================
    # VOLATILIDAD (BASE DEL MODELO)
    # =============================
    df["atr"] = ta.volatility.average_true_range(
        df["high"], df["low"], df["close"], window=14
    )

    df["atr_rel"] = df["atr"] / df["atr"].rolling(50).mean()

    # =============================
    # ESTRUCTURA DE TENDENCIA
    # =============================
    df["ema50"] = ta.trend.ema_indicator(df["close"], window=50)
    df["ema200"] = ta.trend.ema_indicator(df["close"], window=200)

    df["dist_ema50"] = (df["close"] - df["ema50"]) / df["ema50"]
    df["dist_ema200"] = (df["close"] - df["ema200"]) / df["ema200"]

    # Pendiente de EMA50 (estructura real)
    df["slope_ema50"] = df["ema50"].diff(5) / df["ema50"]

    # =============================
    # EXPANSIÓN / COMPRESIÓN
    # =============================
    df["range"] = df["high"] - df["low"]
    df["range_rel"] = df["range"] / df["atr"]

    # =============================
    # VOLUMEN RELATIVO
    # =============================
    df["vol_rel"] = df["volume"] / df["volume"].rolling(50).mean()

    # =============================
    # FUERZA DE TENDENCIA (SUAVE)
    # =============================
    df["trend_strength"] = (
        abs(df["dist_ema50"]) +
        abs(df["dist_ema200"]) +
        abs(df["slope_ema50"])
    )

    # =============================
    # LIMPIEZA FINAL
    # =============================
    df = df.replace([np.inf, -np.inf], np.nan)
    # print("VERIFICACION LEN DF FEATURES:", len(df))
    df = df.dropna().reset_index(drop=True)
    # print("VERIFICACION NOT NAN DF FEATURES:", len(df))
    return df






# =========================================================
# 5️⃣ DATASET SUPERVISADO
# =========================================================
def build_dataset(
    symbols,
    interval="1h",
    limit=350,
    horizon_bars=HORIZON_BARS,          # 4 velas de 1h = 4 horas
    min_move_atr=MIN_MOVE_ATR,          # magnitud mínima favorable (en ATR)
    min_hold_bars=MIN_HOLD_BARS,        # duración mínima del movimiento
    max_adverse_atr=MAX_ADVERSE_ATR,    # drawdown máximo permitido
    max_symbols=MAX_SYMBOLS_TRAIN
):
    """
    Construye dataset ML para detectar REVERSIÓN GRANDE Y DURADERA.
    Coherente con:
    - Scanner en 1H
    - Gestión de trades basada en ATR
    """

    rows = []

    symbols = symbols[:max_symbols]

    for sym in symbols:
        try:
            # 1️⃣ Descargar datos históricos
            df = get_ohlc(sym, interval=interval, limit=limit)
            df = add_features(df)

            if df is None or len(df) < horizon_bars + 50:
                continue

            # 2️⃣ Iterar cada posible punto de entrada
            for i in range(len(df) - horizon_bars):

                row = df.iloc[i]
                future = df.iloc[i + 1 : i + horizon_bars + 1]

                atr = row.atr
                if atr <= 0 or pd.isna(atr):
                    continue

                entry = row.close

                # =========================
                # LONG REVERSAL
                # =========================
                max_favorable = future.high.max()
                min_adverse = future.low.min()

                long_move = (max_favorable - entry) / atr

                long_valid = False

                ###### VERSIÓN 1 (QUE CUMPLA TODAS LAS CONDICIONES DE LONG)
                # long_adverse = (entry - min_adverse) / atr
                # if long_move >= min_move_atr and long_adverse <= max_adverse_atr:

                #     target_price = entry + min_move_atr * atr
                #     hold_bars = future[future.high >= target_price]

                #     if len(hold_bars) >= min_hold_bars:
                #         long_valid = True


                ###### VERSIÓN 2 (QUE CUMPLA 1 DE LAS CONDICIONES DE LONG)
                # long_adverse = (entry - min_adverse) / atr
                # if long_move >= min_move_atr:
                #     score = 0.0
                
                #     if long_adverse <= max_adverse_atr:
                #         score += 1.0

                #     target_price = entry + min_move_atr * atr
                #     hold_bars = future[future.high >= target_price]
                #     if len(hold_bars) >= min_hold_bars:
                #         score += 1.0
                
                #     # etiqueta si cumple lo principal + al menos una condición de calidad
                #     if score >= 1.0:
                #         long_valid = True


                ###### VERSIÓN 3 (CALCULA MÁXIMO RETROCESO ADVERSO HASTA EL PRIMER EVENTO FAVORABLE PARA LONG)
                target_price = entry + min_move_atr * atr
                hit = future[future.high >= target_price]
                
                if len(hit) > 0:
                    cutoff = hit.index[0]
                    adverse_low = future.loc[:cutoff].low.min()
                else:
                    adverse_low = future.low.min()
                
                long_adverse = (entry - adverse_low) / atr

                if long_move >= min_move_atr:
                    score = 0.0
                
                    if long_adverse <= max_adverse_atr:
                        score += 1.0

                    hold_bars = future[future.high >= target_price]
                    if len(hold_bars) >= min_hold_bars:
                        score += 1.0
                
                    # etiqueta si cumple lo principal + al menos una condición de calidad
                    if score >= 1.0:
                        long_valid = True

                
                
                # =========================
                # SHORT REVERSAL
                # =========================
                min_favorable = future.low.min()
                max_adverse = future.high.max()

                short_move = (entry - min_favorable) / atr

                short_valid = False

                ###### VERSIÓN 1 (QUE CUMPLA TODAS LAS CONDICIONES DE SHORT)
                # short_adverse = (max_adverse - entry) / atr
                # if short_move >= min_move_atr and short_adverse <= max_adverse_atr:

                #     target_price = entry - min_move_atr * atr
                #     hold_bars = future[future.low <= target_price]

                #     if len(hold_bars) >= min_hold_bars:
                #         short_valid = True

                ###### VERSIÓN 2 (QUE CUMPLA 1 DE LAS CONDICIONES DE SHORT)
                # short_adverse = (max_adverse - entry) / atr
                # if short_move >= min_move_atr:
                #     score = 0.0
                
                #     if short_adverse <= max_adverse_atr:
                #         score += 1.0

                #     target_price = entry - min_move_atr * atr
                #     hold_bars = future[future.low <= target_price]
                #     if len(hold_bars) >= min_hold_bars:
                #         score += 1.0
                
                #     # etiqueta si cumple lo principal + al menos una condición de calidad
                #     if score >= 1.0:
                #         short_valid = True


                ###### VERSIÓN 3 (CALCULA MÁXIMO RETROCESO ADVERSO HASTA EL PRIMER EVENTO FAVORABLE PARA SHORT)
                target_price = entry - min_move_atr * atr
                hit = future[future.low <= target_price]
                
                if len(hit) > 0:
                    cutoff = hit.index[0]
                    adverse_high = future.loc[:cutoff].high.max()
                else:
                    adverse_high = future.high.max()
                
                short_adverse = (adverse_high - entry) / atr

                if short_move >= min_move_atr:
                    score = 0.0
                
                    if short_adverse <= max_adverse_atr:
                        score += 1.0

                    hold_bars = future[future.low <= target_price]
                    if len(hold_bars) >= min_hold_bars:
                        score += 1.0
                
                    # etiqueta si cumple lo principal + al menos una condición de calidad
                    if score >= 1.0:
                        short_valid = True

                
                # =========================
                # LABEL FINAL
                # =========================
                label = int(long_valid or short_valid)
                
                rows.append({
                    **{f: row[f] for f in FEATURES},
                    "label": label,
                    "Symbol": sym,
                    "Timestamp_UTC": row.time,
                    "ATR": atr,
                    "ATR_rel": atr / entry
                })

        except Exception as e:
            print(f"[WARN] Error creando dataset de entrenamiento: {e}")
            continue

    df_out = pd.DataFrame(rows)
    print("Label distribution", df_out["label"].value_counts(dropna=False))
    return df_out.dropna().reset_index(drop=True)





# =========================================================
# 6️⃣ MODELO ML (EXPLICABLE)
# =========================================================
def train_model(
    dataset: pd.DataFrame,
    features: list,
    n_splits: int = 5,
    random_state: int = 42
):
    """
    Entrena el modelo ML para detectar REVERSIÓN GRANDE y DURADERA.

    - Clasificación binaria
    - Validación temporal (TimeSeriesSplit)
    - Diseñado para scanner 1H / 15m según configuración
    """

    # =============================
    # VALIDACIONES DEFENSIVAS
    # =============================
    required_cols = set(features + ["label"])
    missing = required_cols - set(dataset.columns)
    if missing:
        raise ValueError(f"Dataset incompleto, faltan columnas: {missing}")

    X = dataset[features].values
    y = dataset["label"].values

    # if len(dataset) < 200:
    #     raise ValueError("Dataset demasiado pequeño para entrenamiento robusto")
        
    if len(np.unique(y)) < 2:
        raise ValueError("El dataset no contiene ambas clases (0 y 1)")

    # =============================
    # PIPELINE
    # =============================
    model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2",
            C=0.5,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=500,
            random_state=random_state
        ))
    ])

    # =============================
    # VALIDACIÓN TEMPORAL
    # =============================
    tscv = TimeSeriesSplit(n_splits=n_splits)
    auc_scores = []

    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model.fit(X_train, y_train)
        probas = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, probas)
        auc_scores.append(auc)

    print(
        f"ROC-AUC CV (TimeSeriesSplit): "
        f"{np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}"
    )

    # =============================
    # ENTRENAMIENTO FINAL
    # =============================
    model.fit(X, y)

    clf = model.named_steps["clf"]

    metrics = {
        "n_samples": len(dataset),
        "positive_rate": float(y.mean()),
        "intercept": float(clf.intercept_[0]),
        "coefficients": dict(zip(features, clf.coef_[0]))
    }
    
    return model, metrics






# =========================================================
# 7️⃣ SCANNER FINAL (LONG + SHORT)
# =========================================================

# ==============================
# CONFIG
# ==============================
GMT5 = timezone(timedelta(hours=-5))

FEATURES = [
    "atr",
    "atr_rel",
    "dist_ema50",
    "dist_ema200",
    "slope_ema50",
    "range_rel",
    "vol_rel",
    "trend_strength"
]



# ==============================
# SCANNER DEFINITIVO
# ==============================
from datetime import datetime, timezone
import pandas as pd


def scan_market_ml(
    model,
    symbols,
    interval="1h",
    limit=300,
    min_prob=0.75
):
    """
    Scanner ML para identificar CONTEXTO de reversión
    GRANDE y DURADERA (NO timing).

    Retorna símbolos candidatos para confirmación en 15m.
    """

    results = []
    scan_time = datetime.now(timezone.utc)

    for symbol in symbols:
        try:
            # =============================
            # 1️⃣ Descargar datos
            # =============================
            df = get_ohlc(symbol, interval=interval, limit=limit)
            if df is None or len(df) < 200:
                continue

            df = add_features(df)
            if df is None or df.empty:
                continue

            last = df.iloc[-1]

            # =============================
            # 2️⃣ Validar FEATURES
            # =============================
            if any(pd.isna(last[f]) for f in FEATURES):
                continue

            X = pd.DataFrame(
                [[last[f] for f in FEATURES]],
                columns=FEATURES
            )

            # =============================
            # 3️⃣ Inferencia ML
            # =============================
            prob_reversal = float(
                model.predict_proba(X)[0, 1]
            )

            if prob_reversal < min_prob:
                continue

            # =============================
            # 4️⃣ Dirección NO-ML (interpretativa)
            # =============================
            if last.dist_ema50 < 0:
                direction = "LONG"
            else:
                direction = "SHORT"

            # =============================
            # 5️⃣ Guardar resultado
            # =============================
            results.append({
                "Symbol": symbol,
                "Direction": direction,
                "Prob_Reversal": round(prob_reversal, 4),

                # --- Contexto estructural ---
                "ATR": round(float(last.atr), 6),
                "ATR_rel": round(float(last.atr_rel), 4),
                "Dist_EMA50": round(float(last.dist_ema50), 4),
                "Dist_EMA200": round(float(last.dist_ema200), 4),
                "Trend_Strength": round(float(last.trend_strength), 3),

                # --- Precio base ---
                "Reference_Price": round(float(last.close), 6),

                # --- Tiempo ---
                "Candle_Time_UTC": last.time,
                "Scan_Time_UTC": scan_time
            })

        except Exception as e:
            print(f"[WARN] scan_market_ml {symbol}: {e}")
            continue

    if not results:
        return pd.DataFrame()

    df_out = pd.DataFrame(results)

    # =============================
    # 6️⃣ Ranking final
    # =============================
    df_out = df_out.sort_values(
        "Prob_Reversal",
        ascending=False
    ).reset_index(drop=True)

    return df_out


In [3]:
def main_train_model(HORIZON: int):

    # ==========================================================
    # CONFIGURACIÓN GLOBAL
    # ==========================================================
    INTERVAL = "1h"                 # coherente con scanner 1H
    LIMIT = 350                     # ~15 días de contexto
    # MAX_SYMBOLS = 70                # cobertura + generalización

    # Objetivo del modelo (reversiones grandes y duraderas)
    HORIZON_BARS = HORIZON                # 4 velas de 1h = 4 horas
    MIN_MOVE_ATR = 1.0              # magnitud mínima
    MIN_HOLD_BARS = 1               # ≥ 1 hora sosteniendo el move
    MAX_ADVERSE_ATR = 0.8            # drawdown máximo permitido
    

    RANDOM_STATE = 42
    MODEL_PATH = "reversal_model.joblib"

    print("🚀 Iniciando entrenamiento del modelo de reversión ML")

    # ==========================================================
    # 1️⃣ OBTENER UNIVERSO DE SÍMBOLOS
    # ==========================================================
    all_symbols = get_all_usdt_futures_symbols()
    print(f"Total símbolos USDT Futures: {len(all_symbols)}")

    # ==========================================================
    # 2️⃣ FILTRADO PREVIO (LIQUIDEZ + VOLATILIDAD)
    # ==========================================================
    symbols = filter_symbols_pre_ml(
        symbols=all_symbols,
        interval=INTERVAL,
        max_symbols=MAX_SYMBOLS_TRAIN
    )

    print(f"Símbolos seleccionados para entrenamiento: {len(symbols)}")

    if len(symbols) < 10:
        raise RuntimeError("❌ Muy pocos símbolos tras el filtrado previo")

    # ==========================================================
    # 3️⃣ CONSTRUIR DATASET ML
    # ==========================================================
    print("📊 Construyendo dataset de entrenamiento...")

    df_dataset = build_dataset(
        symbols=symbols,
        interval=INTERVAL,
        limit=LIMIT,
        horizon_bars=HORIZON_BARS,
        min_move_atr=MIN_MOVE_ATR,
        min_hold_bars=MIN_HOLD_BARS,
        max_adverse_atr=MAX_ADVERSE_ATR
    )

    print(f"Dataset final: {df_dataset.shape[0]} filas")

    if df_dataset.empty or df_dataset["label"].nunique() < 2:
        raise RuntimeError("❌ Dataset inválido o sin variabilidad en labels")

    # ==========================================================
    # 4️⃣ ENTRENAR MODELO
    # ==========================================================
    print("🧠 Entrenando modelo ML...")

    model, metrics = train_model(
        df_dataset,
        features=FEATURES,
        random_state=RANDOM_STATE
    )

    print("📈 Métricas del modelo:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    # ==========================================================
    # 5️⃣ GUARDAR MODELO
    # ==========================================================
    joblib.dump(model, MODEL_PATH)
    print(f"💾 Modelo guardado en: {MODEL_PATH}")

    print("✅ ENTRENAMIENTO FINALIZADO CON ÉXITO")

    return model, metrics


In [4]:
# ==========================================================
# MAIN DE INFERENCIAS
# ==========================================================
def compute_mae_atr(
    symbol: str,
    direction: str,
    interval: str = "15m",
    lookback_bars: int = 64,
    atr_period: int = 14
):
    """
    Calcula el MAE (Maximum Adverse Excursion) reciente en unidades de ATR.
    Diseñado para ranking post-scan y validación de trades.

    Retorna:
        dict {
            "mae_atr": float,
            "avg_adverse": float,
            "max_adverse": float,
            "atr": float
        }
        o None si no es confiable
    """

    try:
        df = get_ohlc(symbol, interval=interval, limit=lookback_bars + atr_period + 5)
        if df is None or len(df) < atr_period + 10:
            return None

        df = df.sort_index()

        # ==========================
        # ATR
        # ==========================
        high = df["high"]
        low = df["low"]
        close = df["close"]
        prev_close = close.shift(1)

        tr = (
            (high - low)
            .combine((high - prev_close).abs(), max)
            .combine((low - prev_close).abs(), max)
        )

        atr_series = tr.rolling(atr_period).mean()
        atr = atr_series.iloc[-1]

        if atr is None or atr <= 0:
            return None

        # ==========================
        # Cálculo MAE
        # ==========================
        adverse_moves = []

        for i in range(len(df) - 1):
            entry = close.iloc[i]

            if direction == "LONG":
                adverse = (entry - low.iloc[i + 1]) / atr
            else:
                adverse = (high.iloc[i + 1] - entry) / atr

            if adverse >= 0:
                adverse_moves.append(adverse)

        if len(adverse_moves) < 10:
            return None

        avg_adverse = float(np.mean(adverse_moves))
        max_adverse = float(np.percentile(adverse_moves, 90))

        return {
            "mae_atr": avg_adverse,
            "avg_adverse": avg_adverse,
            "max_adverse": max_adverse,
            "atr": float(atr)
        }

    except Exception as e:
        print(f"[WARN] compute_mae_atr {symbol}: {e}")
        return None



def main_inferences(model, symbols, min_prob=0.75):
    df_scan = scan_market_ml(model=model, symbols=symbols, min_prob=min_prob)

    if len(df_scan)==0:
        print("En el momento no se encontró ningún símbolo que cumpla con las condiciones de entrenamiento.")
        return None
        
    mae_rows = []
    
    for _, row in df_scan.iterrows():
        mae = compute_mae_atr(
            symbol=row["Symbol"],
            direction=row["Direction"]
        )
    
        if mae is not None:
            mae_rows.append({
                "Symbol": row["Symbol"],
                "Direction": row["Direction"],
                "MAE_ATR": mae["mae_atr"],
                "MAE_P90_ATR": mae["max_adverse"]
            })
    
    df_mae = pd.DataFrame(mae_rows)
    
    df_final = df_scan.merge(
        df_mae,
        on=["Symbol", "Direction"],
        how="left"
    )
    
    # Ranking final sugerido
    df_final["Score"] = (
        df_final["Prob_Reversal"] /
        (1.0 + df_final["MAE_ATR"])
    )
    
    df_final = df_final.sort_values("Score", ascending=False)
    
    return df_final

def local_warning(df: pd.DataFrame()):
    if df is not None:
    
        frequency = 2500  # Set Frequency To 2500 Hertz
        duration = 1000   # Set Duration To 1000 ms == 1 second
    
        winsound.Beep(frequency, duration)
        print("Oportunidades de entrada encontradas!")
        
        # Example of beeping with time delays
        for i in range(3):
            winsound.Beep(500, 200) # Frequency 500 Hz, duration 200ms
            time.sleep(0.5)        # Pause for 0.5 seconds between beeps


| Acción                           | Frecuencia recomendada |
| -------------------------------- | ---------------------- |
| Universo de símbolos             | Flexible (dinámico)    |
| Entrenar modelo                  | Cada 2–4 semanas       |
| Reentrenar por cambio de régimen | Inmediato              |
| Inferencia                       | Tiempo real            |


| Capa                         | Timeframe | Rol                                                              |
| ---------------------------- | --------- | ---------------------------------------------------------------- |
| **Scanner ML**               | **1H**    | Detecta *contextos candidatos*                                   |
| **Modelo ML**                | **15m**   | Decide si el contexto tiene **probabilidad de reversión grande** |
| **Confirmación / ejecución** | 15m       | Timing fino de entrada y gestión                                 |


| MAE_ATR     | Interpretación                   |
| ----------- | -------------------------------- |
| `< 0.3`     | 🔥 Muy limpio (ideal TP parcial) |
| `0.3 – 0.6` | ✅ Aceptable                      |
| `0.6 – 0.9` | ⚠️ Riesgoso                      |
| `> 1.0`     | ❌ Evitar                         |


In [5]:
# =========================================================
# 8️⃣ EJECUCIÓN
# =========================================================
print("🧠 Entrenando modelo…")
model, metrics = main_train_model(HORIZON=HORIZON_BARS)

all_symbols = get_all_usdt_futures_symbols()
print(f"Total símbolos USDT Futures: {len(all_symbols)}")

print("🔍 Filtrando símbolos…")
symbols = filter_symbols_pre_ml(
    symbols=all_symbols,
    interval=INTERVAL,
    max_symbols=MAX_SYMBOLS_TRAIN
)

print("🔍 Escaneando mercado…")
signals = main_inferences(model, symbols, min_prob=0.75)

local_warning(signals)

signals

🧠 Entrenando modelo…
🚀 Iniciando entrenamiento del modelo de reversión ML
Total símbolos USDT Futures: 544
Símbolos seleccionados para entrenamiento: 18
📊 Construyendo dataset de entrenamiento...
Label distribution label
1    1970
0     676
Name: count, dtype: int64
Dataset final: 2646 filas
🧠 Entrenando modelo ML...
ROC-AUC CV (TimeSeriesSplit): 0.595 ± 0.025
📈 Métricas del modelo:
  n_samples: 2646
  positive_rate: 0.7445200302343159
  intercept: 0.043574204722052305
  coefficients: {'atr': np.float64(-0.07203787203839307), 'atr_rel': np.float64(-0.08719810589375644), 'dist_ema50': np.float64(0.6872906159387464), 'dist_ema200': np.float64(-0.11069818057587602), 'slope_ema50': np.float64(-0.6255694125776242), 'range_rel': np.float64(-0.049096324860818316), 'vol_rel': np.float64(0.25258541819154673), 'trend_strength': np.float64(0.16092345076372222)}
💾 Modelo guardado en: reversal_model.joblib
✅ ENTRENAMIENTO FINALIZADO CON ÉXITO
Total símbolos USDT Futures: 544
🔍 Filtrando símbolos…
🔍

## Guardar Modelo en Disco Duro

In [ ]:
current_date = str(datetime.now())[:16].replace(":", "").replace(" ", "_")
model_filename = 'Modelo_reversion_grande_futuros_cripto_' + str(int((HORIZON_BARS*15)/60)) + "H_" + current_date + '.joblib'
joblib.dump(model, model_filename)
print(f"Model saved successfully as {model_filename}")

## Guardar resultados del scanner en CSV en memoria

In [ ]:
def save_scanner_results_to_disk(
    signals: pd.DataFrame,
    output_dir: str,
    INTERVAL: str,
    horizon: int
):
    """
    Guarda los resultados del scanner en un CSV en disco.
    El nombre del archivo incluye la estampa de tiempo de la última vela analizada.

    Returns:
        full_path (str): ruta completa del archivo guardado
    """
    if signals.empty:
        raise ValueError("El dataframe de señales está vacío")

    # Asegurar directorio
    os.makedirs(output_dir, exist_ok=True)

    # Asegurar datetime
    signals["Signal_Time"] = pd.to_datetime(signals["Signal_Time"], utc=True)

    # Guardar parámetros de diseño del modelo
    signals["Timeframe"] = INTERVAL
    signals["Horizon_Eval"] = str(horizon)+"h"

    # Última vela utilizada
    last_candle_time = signals["Signal_Time"].max()
    timestamp_str = last_candle_time.strftime("%Y%m%d_%H%MUTC")

    filename = f"scanner_results_{timestamp_str}.csv"
    full_path = os.path.join(output_dir, filename)

    signals.to_csv(full_path, index=False)

    return full_path


In [ ]:
output_dir = "C:/Users/Lenovo S540/Python_scripts"
full_path = save_scanner_results_to_disk(signals, output_dir, INTERVAL, horizon)

print(f"Archivo guardado en la ruta {full_path}")


# 1. EJECUCIÓN DEL MODELO (INFERENCIA)

## Flujo operativo final

OHLC → add_indicators
     → build_dataset (con nuevo label)
     → train_model
     → predict

     → confirm_entry_15m_realtime
     → evaluate_reversal_potential   ← NUEVO
     → monitor_trade (ajustado)


## Cargar Modelo anterior

In [6]:
model_filename = "Modelo_reversion_grande_futuros_cripto_1.0H_2026-01-27_1057.joblib"

model = joblib.load(model_filename)
print(f"Modelo {model_filename} cargado exitosamente")

Modelo Modelo_reversion_grande_futuros_cripto_1.0H_2026-01-27_1057.joblib cargado exitosamente


## Generar Inferencias

¿Hacia dónde es más probable la reversión? → Dir_Confidence: Claridad direccional de la reversión, confianza relativa (0-1)


¿Cuánto puede moverse en términos absolutos? → ATR: Cuánto se mueve este activo en promedio por vela de 1h en USDT                                                                                                  

< 0.55	Señal ambigua → NO operar

0.55 – 0.65	Reversión posible, timing crítico

0.65 – 0.75	Reversión clara

'> 0.75	Reversión muy bien alineada

¿Qué tan explotable es ese movimiento en términos relativos? → ATR_rel: Qué % del precio puede moverse por hora

In [7]:
all_symbols = get_all_usdt_futures_symbols()
print(f"Total símbolos USDT Futures: {len(all_symbols)}")

print("🔍 Filtrando símbolos…")
symbols = filter_symbols_pre_ml(
    symbols=all_symbols,
    interval=INTERVAL,
    max_symbols=MAX_SYMBOLS_TRAIN
)

print("🔍 Escaneando mercado…")
signals = main_inferences(model, symbols, min_prob=0.75)

local_warning(signals)

signals

Total símbolos USDT Futures: 544
🔍 Filtrando símbolos…
🔍 Escaneando mercado…
En el momento no se encontró ningún símbolo que cumpla con las condiciones de entrenamiento.


# 2. CONFIRMACIÓN DE ENTRADA DE LA ORDEN

## Confirmación de entrada 15m

| Timeframe | Rol                                  |
| --------- | ------------------------------------ |
| **1H**    | Bias del modelo (ML)                 |
| **15m**   | Estructura + validación de reversión |
| **5m**    | Timing fino de entrada               |
| **1m**    | (solo monitor_trade) ejecución / SL  |


| Métrica          | Timeframe | Rol                                           |
| ---------------- | --------- | --------------------------------------------- |
| `impulse`        | **15m**   | Validar magnitud real de reversión            |
| `retrace`        | **15m**   | Evitar entrada en vela agotada                |
| `follow_through` | **5m**    | Confirmar que el precio *ya se está moviendo* |
| `MACD energy`    | **5m**    | Timing fino                                   |
| `ATR`            | **15m**   | Ancla de riesgo                               |


In [ ]:
TELEGRAM_BOT_TOKEN = "8593522900:AAFSgAMZAKolZaYqm2GjeqY4Hr3tP7Jnk2M"
TELEGRAM_CHAT_ID = "1733579121"

def send_telegram_alert(message: str):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "Markdown"
    }
    try:
        requests.post(url, json=payload, timeout=5)
    except Exception as e:
        print(f"[WARN] Telegram alert failed: {e}")


In [ ]:
def df15_provider(symbol: str, limit: int = 120):
    """
    Provee dataframe OHLCV de 15m con indicadores técnicos
    necesarios para confirmación y monitoreo.

    Retorna:
        pd.DataFrame o None
    """

    df = get_ohlc(symbol, interval="15m", limit=limit)

    if df is None or len(df) < 50:
        return None

    df = df.sort_index()

    # ---------------- INDICADORES ---------------- #
    df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi"] = 100 - (100 / (1 + rs))

    ema_fast = df["close"].ewm(span=12, adjust=False).mean()
    ema_slow = df["close"].ewm(span=26, adjust=False).mean()

    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=9, adjust=False).mean()

    df["macd_hist"] = macd - macd_signal

    return df


In [ ]:
def compute_atr_from_df(df: pd.DataFrame, period: int = 14):
    if len(df) < period + 1:
        return None

    df = df.copy()
    df["prev_close"] = df["close"].shift(1)

    tr = np.maximum.reduce([
        df["high"] - df["low"],
        (df["high"] - df["prev_close"]).abs(),
        (df["low"] - df["prev_close"]).abs()
    ])

    atr = pd.Series(tr).rolling(period).mean().iloc[-1]

    if pd.isna(atr) or atr <= 0:
        return None

    return float(atr)


In [ ]:
def compute_macd_histogram(
    df: pd.DataFrame,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9,
    price_col: str = "close"
) -> pd.Series:
    """
    Calcula histograma MACD clásico (MACD - Signal).

    Diseñado para:
    - Timing fino de entrada (5m)
    - Evaluar energía + aceleración
    - Sin repainting (solo usa datos cerrados)

    Retorna:
        pd.Series alineada con df.index
    """

    if df is None or len(df) < slow + signal + 5:
        return pd.Series(dtype="float64")

    price = df[price_col].astype(float)

    # =========================
    # EMAs
    # =========================
    ema_fast = price.ewm(span=fast, adjust=False).mean()
    ema_slow = price.ewm(span=slow, adjust=False).mean()

    # =========================
    # MACD
    # =========================
    macd = ema_fast - ema_slow
    signal_line = macd.ewm(span=signal, adjust=False).mean()

    # =========================
    # HISTOGRAMA
    # =========================
    macd_hist = macd - signal_line

    return macd_hist


In [ ]:
def get_macd_energy_threshold(
    atr_15m: float,
    price: float,
    macd_hist_series: pd.Series
):
    """
    Umbral dinámico + confirmación de encendido.
    Diseñado para entradas más tempranas con mejor R/R.
    """

    atr_rel = atr_15m / price

    # ============================
    # BASE THRESHOLD POR VOLATILIDAD
    # ============================
    if atr_rel < 0.0018:        # BTC / ETH muy calmados
        base_th = 0.30
    elif atr_rel < 0.0035:      # volatilidad media
        base_th = 0.22
    else:                       # alts volátiles
        base_th = 0.15

    # ============================
    # AJUSTE POR FASE DEL MOVIMIENTO
    # ============================
    if len(macd_hist_series) < 4:
        return base_th

    h0 = abs(macd_hist_series.iloc[-1])
    h1 = abs(macd_hist_series.iloc[-2])
    h2 = abs(macd_hist_series.iloc[-3])

    accelerating = (h0 > h1 > h2)

    # Si está acelerando → permitimos entrada más temprana
    if accelerating:
        return base_th * 0.75

    return base_th


In [ ]:
def df5_provider(symbol: str, limit: int = 120):
    """
    Descarga velas de 5 minutos desde Binance (API pública),
    retorna DataFrame limpio y ordenado.

    Columnas:
        time (UTC datetime)
        open
        high
        low
        close
        volume

    Retorna:
        pd.DataFrame | None
    """

    try:
        df = get_ohlc(symbol, interval="5m", limit=limit)

        if df is None or len(df) < 50:
            return None


        # Eliminar vela abierta (NO cerrada)
        #now_utc = pd.Timestamp.utcnow().tz_localize("UTC")
        now_utc = datetime.now(timezone.utc)
        if df.iloc[-1]["time"] + pd.Timedelta(minutes=5) > now_utc:
            df = df.iloc[:-1]

        return df

    except Exception as e:
        print(f"[WARN] df5_provider {symbol}: {e}")
        return None


In [ ]:
def check_follow_through_5m(df5, direction: str):
    """
    Confirma continuación mínima en 5m.
    NO redefine reversión, solo evita entrar en pullback inmediato.
    """

    if df5 is None or len(df5) < 3:
        return False

    last = df5.iloc[-1]
    prev = df5.iloc[-2]

    if direction == "LONG":
        print(f"{last.time}: last.close ({last.close}) > prev.open ({prev.open}), prev.close ({prev.close}), last.open ({last.open})")
        return (
            last.close > max(prev.open, prev.close) and
            last.close > last.open
        )
    else:    #SHORT
        print(f"{last.time}: last.close ({last.close}) < prev.open ({prev.open}), prev.close ({prev.close}), last.open ({last.open})")
        return (
            last.close < min(prev.open, prev.close) and
            last.close < last.open
        )


In [ ]:
def confirm_entry_15m_realtime(
    symbol: str,
    direction: str,
    df15_provider,
    df5_provider,
    mode: str = "early",                 # "early" | "conservative"
    poll_seconds: int = 30,
    max_wait_minutes: int = 30
):
    """
    Confirma entrada en tiempo real.
    - Estructura: 15m
    - Timing fino: 5m
    - Riesgo: ATR 15m

    Retorna:
        dict | None
    """

    assert mode in ["early", "conservative"]
    assert direction in ["LONG", "SHORT"]
   
    start_time = datetime.now(timezone.utc)
    max_wait = timedelta(minutes=max_wait_minutes)

    last_checked_candle_time = None
    atr_15m = None

    # =============================
    # PARÁMETROS POR MODO
    # =============================
    if mode == "early":
        min_impulse_atr = 0.6
        max_retrace_atr = 0.35
        min_range_rel = 0.002
    else:  # conservative
        min_impulse_atr = 0.9
        max_retrace_atr = 0.25
        min_range_rel = 0.0035

    msg = f"🟡 *MONITOREANDO ENTRADA*\n {symbol} {direction}\n Modo: {mode}"
    send_telegram_alert(msg)
    print(msg)

    # =============================
    # LOOP PRINCIPAL
    # =============================
    while datetime.now(timezone.utc) - start_time < max_wait:

        try:
            # 1️⃣ Descargar últimas velas 15m
            df15 = df15_provider(symbol, limit=120)
            if df15 is None or len(df15) < 50:
                time.sleep(poll_seconds)
                continue

            last15 = df15.iloc[-1]
            prev15 = df15.iloc[-2]

            # 2️⃣ Detectar nueva vela cerrada
            print(f"\nlast time at {datetime.now()}: {last15.time}")
            print(f"last_checked_candle_time at {datetime.now()}: {last_checked_candle_time}")
                
            if last_checked_candle_time != last15.time:
                atr_15m = compute_atr_from_df(df15, period=14)
                last_checked_candle_time = last15.time
                print(f"First check atr_15m at {datetime.now()}: {atr_15m}")
                if atr_15m is None or atr_15m <= 0:
                    time.sleep(poll_seconds)
                    continue
            
            # =============================
            # MÉTRICAS DE VELA
            # =============================
            candle_range = last15.high - last15.low
            range_rel = candle_range / last15.close

            if range_rel < min_range_rel:
                print(f"range_rel at {datetime.now()}: {range_rel}")
                time.sleep(poll_seconds)
                continue

            if direction == "LONG":
                impulse = (last15.close - last15.low) / atr_15m
                retrace = (last15.high - last15.close) / atr_15m
                valid_structure = last15.close > last15.open
            else:
                impulse = (last15.high - last15.close) / atr_15m
                retrace = (last15.close - last15.low) / atr_15m
                valid_structure = last15.close < last15.open

            print(f"impulse at {datetime.now()}: {impulse}")
            print(f"retrace at {datetime.now()}: {retrace}")
            print(f"valid_structure at {datetime.now()}: {valid_structure}")

            
            # FILTROS ANTI-ENTRADA PREMATURA
            # =============================
            # CONDICIONES DE ENTRADA POR IMPULSO, RETROCESO Y ESTRUCTURA
            # =============================
            if not (
                impulse >= min_impulse_atr and
                retrace <= max_retrace_atr and
                valid_structure
            ):
                time.sleep(poll_seconds)
                continue
            
            # =============================
            # FILTRO DE TIMING FINO (5m)
            # =============================
            # --------CONDICIÓN CONTINUIDAD ESTRUCTURAL (TRIGGER) 5m FOLLOW-THROUGH --------
            df5 = df5_provider(symbol, limit=120)
            if df5 is None or len(df5) < 50:
                time.sleep(poll_seconds)
                continue
            if not check_follow_through_5m(df5, direction):
                print(f"check_follow_through_5m: FALSE")
                time.sleep(poll_seconds)
                continue


            # --------CONDICIÓN VOLUMEN MACD (TRIGGER) 5m MACD ENERGY --------           
            macd_hist = compute_macd_histogram(df5)
            # Umbral dinámico según volatilidad y tendencia de MACD
            macd_energy = abs(macd_hist.iloc[-1])
            
            macd_energy_threshold = get_macd_energy_threshold(
                atr_15m=atr_15m,
                price=last15.close,
                macd_hist_series=macd_hist
            )
            
            if macd_energy < macd_energy_threshold:
                print(
                    f"MACD energy insuficiente: {macd_energy:.2f} "
                    f"(threshold {macd_energy_threshold:.2f})\n"
                )
                time.sleep(poll_seconds)
                continue

            
            # =============================
            # ENTRADA CONFIRMADA
            # =============================            
            msg = f"🟢 *SEÑAL DE ENTRADA CONFIRMADA*\n \
                {symbol} {direction}\n \
                Modo: {mode}\n \
                Precio: `{round(last15.close, 6)}`\n \
                ATR 15m: `{round(atr_15m, 6)}`\n \
                Impulso: `{round(impulse, 2)} ATR`\n \
                Retroceso: `{round(retrace, 2)} ATR`"
            send_telegram_alert(msg)
            print(msg)

            return {
                "symbol": symbol,
                "direction": direction,
                "price": last15.close,
                "atr_15m": atr_15m,
                "mode": mode,
                "timestamp": last15.time
            }


        except Exception as e:
            print(f"[WARN] confirm_entry_15m_realtime {symbol}: {e}")
            time.sleep(poll_seconds)

    # =============================
    # TIMEOUT
    # =============================
    msg = f"⏰ *ENTRADA NO CONFIRMADA*\n {symbol} {direction}\n Tiempo máximo agotado ({max_wait_minutes} min)"
    send_telegram_alert(msg)
    print(msg)
    return None


In [ ]:
posicion = 0

# symbol = signals.loc[posicion, "Symbol"]
# direction = signals.loc[posicion, "Direction"]

symbol = "VICUSDT"
direction = "SHORT"


print("symbol:", symbol)
print("direction:", direction)

In [ ]:
entry = confirm_entry_15m_realtime(
    symbol=symbol,
    direction=direction,
    df15_provider=df15_provider,
    df5_provider=df5_provider,
    mode="early",
    max_wait_minutes=30
)


print(entry)
local_warning(entry)

# 3. CONFIRMACIÓN DE SALIDA DE LA ORDEN

## Confirmación de salida 15m

In [ ]:
def df1m_provider(
    symbol: str,
    limit: int = 50 # Límite mínimo habilitado en el API Binance es 50
):
    """
    Proveedor OHLC 1m para ejecución de trade.

    Uso exclusivo:
    - Verificación precisa de SL / TP / TP parcial
    - Precio más reciente disponible

    Retorna:
    - DataFrame ordenado por tiempo ascendente
    - Columnas mínimas: time, open, high, low, close
    """

    try:
        df = get_ohlc(
            symbol=symbol,
            interval="1m",
            limit=limit
        )

        if df is None or df.empty:
            return None

        # Normalización defensiva
        df = df.copy()

        # Asegurar tipos numéricos
        for col in ["open", "high", "low", "close"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df["time"] = pd.to_datetime(df["time"], utc=True)

        # Limpiar datos inválidos
        df = df.dropna(subset=["close", "high", "low"])

        if len(df) == 0:
            return None

        # Orden temporal estricto
        df = df.sort_values("time").reset_index(drop=True)

        return df[["time", "open", "high", "low", "close"]]

    except Exception as e:
        print(f"[WARN] df1m_provider {symbol}: {e}")
        return None


In [ ]:
def structural_exit_conditions(
    last,                      # pd.Series (vela cerrada)
    direction: str,             # "LONG" | "SHORT"
    atr_rel: float              # atr_15m / price (ya calculado afuera)
) -> bool:
    """
    Detecta debilitamiento estructural del trade usando SOLO:
    - estructura de vela
    - relación rango / ATR
    - rechazo contra la dirección

    Retorna:
        True  -> estructura inválida (contabilizar para salida)
        False -> estructura válida
    """

    # =============================
    # VALIDACIONES DEFENSIVAS
    # =============================
    if last is None or atr_rel <= 0:
        return False

    # =============================
    # MÉTRICAS DE VELA
    # =============================
    candle_range = last.high - last.low
    if candle_range <= 0:
        return True

    body = abs(last.close - last.open)
    body_rel = body / candle_range

    upper_wick = last.high - max(last.close, last.open)
    lower_wick = min(last.close, last.open) - last.low

    range_rel = candle_range / max(last.close, 1e-9)

    # =============================
    # FILTROS DE RUIDO
    # =============================
    if range_rel < 0.001:
        return False

    # =============================
    # REGLAS ESTRUCTURALES
    # =============================
    invalid = False

    if direction == "LONG":
        # Rechazo fuerte hacia arriba
        if upper_wick / candle_range > 0.6:
            invalid = True

        # Cuerpo débil + cierre bajo
        if body_rel < 0.25 and last.close < last.open:
            invalid = True

        # Volatilidad se colapsa
        if candle_range < 0.4 * atr_rel * last.close:
            invalid = True

    else:  # SHORT
        # Rechazo fuerte hacia abajo
        if lower_wick / candle_range > 0.6:
            invalid = True

        # Cuerpo débil + cierre alto
        if body_rel < 0.25 and last.close > last.open:
            invalid = True

        # Volatilidad se colapsa
        if candle_range < 0.4 * atr_rel * last.close:
            invalid = True

    return invalid


In [ ]:
def calculate_position_plan(
    entry_price: float,
    atr_15m: float,
    direction: str,
    target_usd_tp_partial: float,
    tp_partial_mult: float,
    sl_mult: float,
    max_leverage: int = 30,
    min_capital: float = 100.0,
    max_capital: float = 300.0,
    max_risk_usd: float = 80.0
):
    """
    Calcula capital y leverage óptimos basados en ATR y objetivo USD.

    Retorna:
        dict
    """

    # --------------------------------------------------
    # Movimiento esperado (TP parcial)
    # --------------------------------------------------
    tp_partial_move = tp_partial_mult * atr_15m
    tp_partial_pct = tp_partial_move / entry_price

    if tp_partial_pct <= 0:
        return None

    # --------------------------------------------------
    # Exposición necesaria para lograr target USD
    # --------------------------------------------------
    exposure_needed = target_usd_tp_partial / tp_partial_pct

    # --------------------------------------------------
    # Stop Loss en USD por unidad de exposición
    # --------------------------------------------------
    sl_move = sl_mult * atr_15m
    sl_pct = sl_move / entry_price

    # Capital mínimo para no violar riesgo
    capital_by_risk = max_risk_usd / sl_pct

    # Capital final limitado
    capital = max(min_capital, min(capital_by_risk, max_capital))

    # Leverage requerido
    leverage = exposure_needed / capital
    leverage = min(max(1.0, leverage), max_leverage)

    # Exposición final
    exposure = capital * leverage

    return {
        "capital_usd": round(capital, 2),
        "leverage": round(leverage, 1),
        "exposure_usd": round(exposure, 2),
        "tp_partial_usd": round(exposure * tp_partial_pct, 2),
        "sl_usd": round(exposure * sl_pct, 2)
    }


In [ ]:
def monitor_trade(
    symbol: str,
    direction: str,
    entry_price: float,
    atr_15m: float,
    horizon: int,  # Cantidad de velas de 15 minutos para las que se espera la reversión
    end_time_utc,
    df15_provider,
    df1m_provider,
    poll_seconds: int = 30
):
    """
    Monitorea un trade abierto separando:
    - 15m → estructura, trailing, BE, salida estructural
    - 1m  → ejecución real de SL / TP

    No ejecuta órdenes, solo gestiona alertas de SALIDAS por Telegram.
    """
    """
    El loop FINALIZA SOLO si:
    - Stop Loss alcanzado
    - Take Profit final alcanzado
    - Tiempo máximo expirado
    """

    # ==========================================================
    # CONFIGURACIÓN POR HORIZONTE (VELAS 15m)
    # ==========================================================
    if horizon >= 4:
        sl_mult = 1.2
        tp_partial_mult = 0.9
        be_mult = 1.0
        tp_mult = 2.5
        trail_mult = 1.2
    elif horizon == 2:
        sl_mult = 0.9
        tp_partial_mult = 0.7
        be_mult = 0.8
        tp_mult = 1.8
        trail_mult = 1.0
    else:
        sl_mult = 0.7
        tp_partial_mult = 0.5
        be_mult = 0.6
        tp_mult = 1.2
        trail_mult = 0.8

    # ==========================================================
    # ESTADO INTERNO
    # ==========================================================
    tp_partial_alerted = False
    be_alerted = False
    structural_exit_alerted = False
    trailing_last_sl = None
    invalid_structure_count = 0

    last_be_candle_time = None
    last_15m_candle_time = None

    # ==========================================================
    # NIVELES INICIALES
    # ==========================================================
    if direction == "LONG":
        sl = max(entry_price - sl_mult * atr_15m, 0.0)
        tp = entry_price + tp_mult * atr_15m
        tp_partial = entry_price + tp_partial_mult * atr_15m
    else:
        sl = entry_price + sl_mult * atr_15m
        tp = entry_price - tp_mult * atr_15m
        tp_partial = entry_price - tp_partial_mult * atr_15m

    msg = (
        f"🟢 *TRADE ABIERTO*\n"
        f"Symbol: `{symbol}`\n"
        f"Direction: *{direction}*\n"
        f"Entry: `{round(entry_price,6)}`\n"
        f"SL inicial: `{round(sl,6)}`\n"
        f"TP parcial: `{round(tp_partial,6)}`\n"
        f"TP final: `{round(tp,6)}`\n"
    )
    send_telegram_alert(msg)
    print(msg)

    
    # ==========================================================
    # PLAN DE POSICIÓN RECOMENDADO (USD + LEVERAGE)
    # ==========================================================
    position_plan = calculate_position_plan(
        entry_price=entry_price,
        atr_15m=atr_15m,
        direction=direction,
        target_usd_tp_partial=25.0,     # 🎯 objetivo TP parcial
        tp_partial_mult=tp_partial_mult,
        sl_mult=sl_mult
    )

    if position_plan:
        msg = (
            f"📐 *PLAN DE POSICIÓN RECOMENDADO*\n"
            f"Capital: `{position_plan['capital_usd']} USD`\n"
            f"Leverage: `{position_plan['leverage']}x`\n"
            f"Exposición: `{position_plan['exposure_usd']} USD`\n\n"
            f"SL estimado: `-{position_plan['sl_usd']} USD`\n"
            f"TP parcial: `+{position_plan['tp_partial_usd']} USD`\n"
            f"TP final: `~{round(position_plan['tp_partial_usd'] * (tp_mult / tp_partial_mult), 2)} USD`"
            f"atr_15m: `{atr_15m}`"
        )
        send_telegram_alert(msg)
        print(msg)


    # ==========================================================
    # LOOP PRINCIPAL
    # ==========================================================
    while datetime.now(timezone.utc) < end_time_utc:

        try:
            # ==================================================
            # 1️⃣ PRECIO REAL (1m) → EJECUCIÓN
            # ==================================================
            df1m = df1m_provider(symbol)
            if df1m is None or len(df1m) < 1:
                time.sleep(poll_seconds)
                continue

            last_price = float(df1m.iloc[-1].close)
            print(f" ({datetime.now()}) last_price: {last_price}\n")
            # --------------------------
            # STOP LOSS (EJECUCIÓN REAL)
            # --------------------------
            if direction == "LONG" and last_price <= sl:
                msg = f"🔴 *STOP LOSS ALCANZADO*\n{symbol} LONG\nNivel: `{round(sl,6)}`"
                send_telegram_alert(msg)
                print(msg)
                return {"Status": "STOP_LOSS", "Level": sl}

            if direction == "SHORT" and last_price >= sl:
                msg = f"🔴 *STOP LOSS ALCANZADO*\n{symbol} SHORT\nNivel: `{round(sl,6)}`"
                send_telegram_alert(msg)
                print(msg)
                return {"Status": "STOP_LOSS", "Level": sl}

            # --------------------------
            # TP PARCIAL
            # --------------------------
            if not tp_partial_alerted:
                if (direction == "LONG" and last_price >= tp_partial) or \
                   (direction == "SHORT" and last_price <= tp_partial):

                    tp_partial_alerted = True
                    msg = f"📈 *TP PARCIAL ALCANZADO*\n{symbol} {direction}\nNivel: `{round(tp_partial,6)}`"
                    send_telegram_alert(msg)
                    print(msg)

            # --------------------------
            # TP FINAL
            # --------------------------
            if (direction == "LONG" and last_price >= tp) or \
               (direction == "SHORT" and last_price <= tp):

                msg = f"🟢 *TAKE PROFIT FINAL ALCANZADO*\n{symbol} {direction}\nNivel: `{round(tp,6)}`"
                send_telegram_alert(msg)
                print(msg)
                return {"Status": "TAKE_PROFIT", "Level": tp}

            # ==================================================
            # 2️⃣ ESTRUCTURA (15m) → LÓGICA
            # ==================================================
            df15 = df15_provider(symbol)
            if df15 is None or len(df15) < 2:
                time.sleep(poll_seconds)
                continue

            last15 = df15.iloc[-1]
            prev15 = df15.iloc[-2]

            # Detectar nueva vela 15m cerrada
            if last_15m_candle_time != last15.time:
                last_15m_candle_time = last15.time

                # --------------------------
                # BREAK EVEN
                # --------------------------
                if not be_alerted:
                    if (direction == "LONG" and last15.high >= entry_price + be_mult * atr_15m) or \
                       (direction == "SHORT" and last15.low <= entry_price - be_mult * atr_15m):

                        sl = entry_price
                        be_alerted = True
                        last_be_candle_time = last15.time

                        msg = f"🟡 *BREAK EVEN SUGERIDO*\n{symbol} {direction}\nNuevo SL: `{round(sl,6)}`"
                        send_telegram_alert(msg)
                        print(msg)

                # --------------------------
                # TRAILING STOP (solo tras BE)
                # --------------------------
                if be_alerted and last15.time != last_be_candle_time:
                    if direction == "LONG":
                        new_sl = max(last15.high - trail_mult * atr_15m, sl)
                        if trailing_last_sl is None or new_sl > trailing_last_sl:
                            trailing_last_sl = new_sl
                            sl = new_sl
                            msg = f"🔁 *TRAILING STOP SUGERIDO*\n{symbol} LONG\nNuevo SL: `{round(sl,6)}`"
                            send_telegram_alert(msg)
                            print(msg)
                    else:
                        new_sl = min(last15.low + trail_mult * atr_15m, sl)
                        if trailing_last_sl is None or new_sl < trailing_last_sl:
                            trailing_last_sl = new_sl
                            sl = new_sl
                            msg = f"🔁 *TRAILING STOP SUGERIDO*\n{symbol} SHORT\nNuevo SL: `{round(sl,6)}`"
                            send_telegram_alert(msg)
                            print(msg)

                # --------------------------
                # SALIDA ESTRUCTURAL
                # --------------------------
                atr_rel = atr_15m / max(last15.close, 1e-9)

                if structural_exit_conditions(last15, direction, atr_rel):
                    invalid_structure_count += 1
                else:
                    invalid_structure_count = max(0, invalid_structure_count - 1)

                if invalid_structure_count >= 2 and not structural_exit_alerted:
                    structural_exit_alerted = True
                    msg = f"⚠️ *SALIDA ESTRUCTURAL SUGERIDA*\n{symbol} {direction}"
                    send_telegram_alert(msg)
                    print(msg)

            time.sleep(poll_seconds)

        except Exception as e:
            print(f"[WARN] monitor_trade {symbol}: {e}")
            time.sleep(poll_seconds)

    # ==========================================================
    # TIEMPO EXPIRADO
    # ==========================================================
    msg = f"⏰ *TIEMPO DE TRADE EXPIRADO*\n{symbol} {direction}\nCierre manual recomendado"
    send_telegram_alert(msg)
    print(msg)

    return {"Status": "TIME_EXIT"}


In [ ]:
print(entry)

In [ ]:
# ---------------- PARÁMETROS DEL TRADE ---------------- #

#symbol = entry["symbol"]
#direction = entry["direction"]
#atr_15m = entry["atr_15m"]
symbol = "KITEUSDT"
direction = "LONG"
atr_15m = 0.003383   # ATR calculado en 15m en el momento de entrada calculado por la función "confirm_entry_15m_realtime" 

horizon = HORIZON_BARS  # Horizonte de tiempo para la reversión entrenada en el modelo de escaneo # Cantidad de velas de 15 minutos para las que se espera la reversión
horizon = 2

last_price = get_ohlc(symbol=symbol, interval="1m", limit=50)
last_price = last_price.iloc[-1, -2]

entry_price = 0.18933 # Precio de entrada REAL (ejecutado por ti)
#entry_price = last_price


# Tiempo máximo de monitoreo (ej: 4 horas)
end_time_utc = datetime.now(timezone.utc) + timedelta(hours=horizon)

# ---------------- INICIAR MONITOREO ---------------- #

result = monitor_trade(
    symbol=symbol,
    direction=direction,
    entry_price=entry_price,
    atr_15m=atr_15m,
    horizon=horizon,
    end_time_utc=end_time_utc,
    df15_provider=df15_provider,
    df1m_provider=df1m_provider,
    poll_seconds=60  # revisar cada 1 minuto
)

print("Resultado final del trade:", result)


In [ ]:
69563.9 + (0.9*431.2388888888947)

In [ ]:
69563.9 + (0.7*431.2388888888947)

In [ ]:
(69957.9 - 69563.9) / 431.2388888888947